In [ ]:
!pip install -q tensorflow optuna mlflow lime

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import gc
import random
import json
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import optuna
import mlflow
import mlflow.tensorflow

from lime.lime_tabular import LimeTabularExplainer

In [ ]:
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
CLONE_REPO = False

GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/finalterm-machine-learning.git"
LOCAL_REPO_PATH = "/content/finalterm-machine-learning"

if CLONE_REPO:
    !rm -rf /content/finalterm-machine-learning
    !git clone {GITHUB_REPO_URL}
    print("Repository cloned.")
else:
    print("Skipping clone repository.")

In [ ]:
LOCAL_DATA_PATH = "/content/midterm-regresi-dataset.csv"

DRIVE_DATA_PATH = "/content/drive/MyDrive/Midterm ML/midterm-regresi-dataset.csv"

# contoh format raw github:
# https://raw.githubusercontent.com/USERNAME/REPO/main/data/midterm-regresi-dataset.csv
GITHUB_RAW_DATA_URL = "https://raw.githubusercontent.com/YOUR_USERNAME/finalterm-machine-learning/main/data/midterm-regresi-dataset.csv"

CLONED_REPO_DATA_PATH = "/content/finalterm-machine-learning/data/midterm-regresi-dataset.csv"

In [ ]:
def load_dataset():
    # 1) local file in /content
    if os.path.exists(LOCAL_DATA_PATH):
        print(f"Loading dataset from LOCAL path: {LOCAL_DATA_PATH}")
        return pd.read_csv(LOCAL_DATA_PATH, header=None)

    # 2) google drive
    if os.path.exists(DRIVE_DATA_PATH):
        print(f"Loading dataset from DRIVE path: {DRIVE_DATA_PATH}")
        return pd.read_csv(DRIVE_DATA_PATH, header=None)

    # 3) cloned repo path
    if os.path.exists(CLONED_REPO_DATA_PATH):
        print(f"Loading dataset from CLONED REPO path: {CLONED_REPO_DATA_PATH}")
        return pd.read_csv(CLONED_REPO_DATA_PATH, header=None)

    # 4) github raw url
    try:
        print(f"Trying to load dataset from GitHub RAW URL:\n{GITHUB_RAW_DATA_URL}")
        return pd.read_csv(GITHUB_RAW_DATA_URL, header=None)
    except Exception as e:
        raise FileNotFoundError(
            "Dataset tidak ditemukan.\n"
            "Cek LOCAL_DATA_PATH / DRIVE_DATA_PATH / CLONED_REPO_DATA_PATH / GITHUB_RAW_DATA_URL.\n"
            f"Error detail: {e}"
        )

df = load_dataset()
print("Dataset shape:", df.shape)
df.head()

In [ ]:
target_col = "Year"
feature_names = [f"feature_{i}" for i in range(1, df.shape[1])]
df.columns = [target_col] + feature_names

print("Total columns:", len(df.columns))
df.head()

In [ ]:
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.astype(np.float32)

print(df.dtypes.head())
print("Memory usage (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 2))

In [ ]:
df.info()

In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

In [ ]:
missing_summary = df.isnull().sum().sort_values(ascending=False)
print("Total missing values:", df.isnull().sum().sum())
missing_summary.head(20)

In [ ]:
if df.isnull().sum().sum() > 0:
    df = df.fillna(df.median(numeric_only=True))
    print("Missing values after imputation:", df.isnull().sum().sum())
else:
    print("No missing values detected.")

In [ ]:
df[target_col].describe()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df[target_col], bins=30, kde=True)
plt.title("Distribution of Song Release Year")
plt.xlabel("Year")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x=df[target_col])
plt.title("Boxplot of Target (Year)")
plt.show()

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

print("Feature shape:", X.shape)
print("Target shape :", y.shape)

In [ ]:
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED
)

print("Train:", X_train_df.shape, y_train.shape)
print("Test :", X_test_df.shape, y_test.shape)

In [ ]:
lower_q = X_train_df.quantile(0.01)
upper_q = X_train_df.quantile(0.99)

X_train_df = X_train_df.clip(lower=lower_q, upper=upper_q, axis=1)
X_test_df = X_test_df.clip(lower=lower_q, upper=upper_q, axis=1)

print("Outlier clipping completed.")

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_df).astype(np.float32)
X_test_scaled = scaler.transform(X_test_df).astype(np.float32)

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled test shape :", X_test_scaled.shape)

In [ ]:
k_best = min(60, X_train_scaled.shape[1])

selector = SelectKBest(score_func=f_regression, k=k_best)
X_train_kbest = selector.fit_transform(X_train_scaled, y_train)
X_test_kbest = selector.transform(X_test_scaled)

selected_mask = selector.get_support()
selected_features = X.columns[selected_mask].tolist()

print("Original features:", X_train_scaled.shape[1])
print("Selected features:", X_train_kbest.shape[1])
print("Example selected features:", selected_features[:10])

In [ ]:
pca = PCA(n_components=0.95, random_state=SEED)

X_train_pca = pca.fit_transform(X_train_kbest).astype(np.float32)
X_test_pca = pca.transform(X_test_kbest).astype(np.float32)

print("Train shape after PCA:", X_train_pca.shape)
print("Test shape after PCA :", X_test_pca.shape)
print("Total explained variance:", pca.explained_variance_ratio_.sum())

In [ ]:
cum_var = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(8,5))
plt.plot(range(1, len(cum_var)+1), cum_var, marker='o')
plt.axhline(0.95, linestyle='--')
plt.title("Cumulative Explained Variance by PCA")
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.show()

In [ ]:
def build_regression_model(
    input_dim,
    units1=128,
    units2=64,
    dropout1=0.3,
    dropout2=0.2,
    learning_rate=1e-3
):
    model = Sequential([
        Input(shape=(input_dim,)),

        Dense(units1, activation="relu"),
        BatchNormalization(),
        Dropout(dropout1),

        Dense(units2, activation="relu"),
        BatchNormalization(),
        Dropout(dropout2),

        Dense(1, activation="linear")
    ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    verbose=1
)

In [ ]:
baseline_model = build_regression_model(
    input_dim=X_train_pca.shape[1],
    units1=128,
    units2=64,
    dropout1=0.3,
    dropout2=0.2,
    learning_rate=1e-3
)

history_baseline = baseline_model.fit(
    X_train_pca,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=256,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history_baseline.history["loss"], label="Train Loss")
plt.plot(history_baseline.history["val_loss"], label="Validation Loss")
plt.title("Baseline Model Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.show()

In [ ]:
y_pred_baseline = baseline_model.predict(X_test_pca, verbose=0).flatten()

In [ ]:
baseline_mse = mean_squared_error(y_test, y_pred_baseline)
baseline_rmse = np.sqrt(baseline_mse)
baseline_mae = mean_absolute_error(y_test, y_pred_baseline)
baseline_r2 = r2_score(y_test, y_pred_baseline)

print("=== BASELINE MODEL ===")
print("MSE  :", baseline_mse)
print("RMSE :", baseline_rmse)
print("MAE  :", baseline_mae)
print("R2   :", baseline_r2)

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(y_test, y_pred_baseline, alpha=0.4)
plt.xlabel("Actual Year")
plt.ylabel("Predicted Year")
plt.title("Baseline Model: Actual vs Predicted")
plt.show()

In [ ]:
baseline_residuals = y_test - y_pred_baseline

plt.figure(figsize=(7,5))
plt.scatter(y_pred_baseline, baseline_residuals, alpha=0.4)
plt.axhline(0, linestyle='--')
plt.xlabel("Predicted Year")
plt.ylabel("Residual")
plt.title("Baseline Model Residual Plot")
plt.show()

In [ ]:
X_tune, _, y_tune, _ = train_test_split(
    X_train_pca,
    y_train,
    train_size=0.4,
    random_state=SEED
)

print("Optuna tuning subset shape:", X_tune.shape, y_tune.shape)

In [ ]:
def objective(trial):
    units1 = trial.suggest_categorical("units1", [64, 128, 256])
    units2 = trial.suggest_categorical("units2", [32, 64, 128])
    dropout1 = trial.suggest_float("dropout1", 0.1, 0.5)
    dropout2 = trial.suggest_float("dropout2", 0.1, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    model = build_regression_model(
        input_dim=X_tune.shape[1],
        units1=units1,
        units2=units2,
        dropout1=dropout1,
        dropout2=dropout2,
        learning_rate=learning_rate
    )

    es = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        X_tune,
        y_tune,
        validation_split=0.2,
        epochs=15,
        batch_size=batch_size,
        callbacks=[es],
        verbose=0
    )

    best_val_loss = min(history.history["val_loss"])

    tf.keras.backend.clear_session()
    gc.collect()

    return best_val_loss

In [ ]:
study = optuna.create_study(direction="minimize")

study.optimize(
    objective,
    n_trials=10,
    show_progress_bar=True
)

print("Best trial value:", study.best_trial.value)
print("Best params:", study.best_params)

In [ ]:
best_params = study.best_params

best_model = build_regression_model(
    input_dim=X_train_pca.shape[1],
    units1=best_params["units1"],
    units2=best_params["units2"],
    dropout1=best_params["dropout1"],
    dropout2=best_params["dropout2"],
    learning_rate=best_params["learning_rate"]
)

early_stop_best = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr_best = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    verbose=1
)

history_best = best_model.fit(
    X_train_pca,
    y_train,
    validation_split=0.2,
    epochs=40,
    batch_size=best_params["batch_size"],
    callbacks=[early_stop_best, reduce_lr_best],
    verbose=1
)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history_best.history["loss"], label="Train Loss")
plt.plot(history_best.history["val_loss"], label="Validation Loss")
plt.title("Tuned Model Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.show()

In [ ]:
y_pred_best = best_model.predict(X_test_pca, verbose=0).flatten()

In [ ]:
best_mse = mean_squared_error(y_test, y_pred_best)
best_rmse = np.sqrt(best_mse)
best_mae = mean_absolute_error(y_test, y_pred_best)
best_r2 = r2_score(y_test, y_pred_best)

print("=== TUNED MODEL ===")
print("MSE  :", best_mse)
print("RMSE :", best_rmse)
print("MAE  :", best_mae)
print("R2   :", best_r2)

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(y_test, y_pred_best, alpha=0.4)
plt.xlabel("Actual Year")
plt.ylabel("Predicted Year")
plt.title("Tuned Model: Actual vs Predicted")
plt.show()

In [ ]:
best_residuals = y_test - y_pred_best

plt.figure(figsize=(7,5))
plt.scatter(y_pred_best, best_residuals, alpha=0.4)
plt.axhline(0, linestyle='--')
plt.xlabel("Predicted Year")
plt.ylabel("Residual")
plt.title("Tuned Model Residual Plot")
plt.show()

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["Baseline", "Tuned"],
    "MSE": [baseline_mse, best_mse],
    "RMSE": [baseline_rmse, best_rmse],
    "MAE": [baseline_mae, best_mae],
    "R2": [baseline_r2, best_r2]
})

comparison_df

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=comparison_df, x="Model", y="RMSE")
plt.title("Baseline vs Tuned Model RMSE Comparison")
plt.show()

In [ ]:
mlflow.set_experiment("Song Year Prediction - Deep Learning Regression")

with mlflow.start_run(run_name="tuned_regression_model"):
    mlflow.log_param("pipeline", "imputation + scaling + SelectKBest + PCA + DNN")

    for k, v in best_params.items():
        mlflow.log_param(k, v)

    mlflow.log_metric("baseline_mse", float(baseline_mse))
    mlflow.log_metric("baseline_rmse", float(baseline_rmse))
    mlflow.log_metric("baseline_mae", float(baseline_mae))
    mlflow.log_metric("baseline_r2", float(baseline_r2))

    mlflow.log_metric("best_mse", float(best_mse))
    mlflow.log_metric("best_rmse", float(best_rmse))
    mlflow.log_metric("best_mae", float(best_mae))
    mlflow.log_metric("best_r2", float(best_r2))

    mlflow.tensorflow.log_model(best_model, "song_year_regression_model")

In [ ]:
lime_train_size = min(2000, X_train_pca.shape[0])

lime_indices = np.random.choice(
    X_train_pca.shape[0],
    size=lime_train_size,
    replace=False
)

X_train_lime = X_train_pca[lime_indices]

In [ ]:
pca_feature_names = [f"PC_{i+1}" for i in range(X_train_pca.shape[1])]

explainer = LimeTabularExplainer(
    training_data=X_train_lime,
    feature_names=pca_feature_names,
    mode="regression"
)

In [ ]:
exp = explainer.explain_instance(
    data_row=X_test_pca[0],
    predict_fn=lambda x: best_model.predict(x, verbose=0).flatten(),
    num_features=min(10, X_test_pca.shape[1])
)

print("LIME explanation for first test sample:")
for item in exp.as_list():
    print(item)

fig = exp.as_pyplot_figure()
plt.show()

In [ ]:
best_model.save("song_year_regression_best_model.keras")
print("Model saved successfully.")

In [ ]:
final_result = pd.DataFrame({
    "Metric": [
        "Baseline_MSE", "Baseline_RMSE", "Baseline_MAE", "Baseline_R2",
        "Best_MSE", "Best_RMSE", "Best_MAE", "Best_R2"
    ],
    "Value": [
        baseline_mse, baseline_rmse, baseline_mae, baseline_r2,
        best_mse, best_rmse, best_mae, best_r2
    ]
})

final_result.to_csv("regression_metrics_comparison.csv", index=False)
final_result

In [ ]:
prediction_df = pd.DataFrame({
    "Actual_Year": y_test.values,
    "Predicted_Year": y_pred_best
})

prediction_df.to_csv("song_year_predictions.csv", index=False)
prediction_df.head()

In [ ]:
import pickle

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("selector.pkl", "wb") as f:
    pickle.dump(selector, f)

with open("pca.pkl", "wb") as f:
    pickle.dump(pca, f)

print("Scaler, selector, and PCA objects saved.")

In [ ]:
summary_df = pd.DataFrame({
    "Step": [
        "Data Cleaning",
        "Missing Value Handling",
        "Outlier Handling",
        "Scaling",
        "Feature Selection",
        "PCA",
        "Baseline Deep Learning",
        "Optuna Tuning",
        "Final Model Evaluation",
        "MLflow Tracking",
        "LIME Interpretation"
    ],
    "Status": ["Done"] * 11
})

summary_df

In [ ]:
print("KESIMPULAN")
print("- Pipeline regresi deep learning berhasil dibangun untuk memprediksi tahun rilis lagu.")
print("- Tahapan yang dilakukan meliputi preprocessing, imputasi missing values, clipping outlier, scaling, feature selection, PCA, baseline model, Optuna tuning, evaluasi, MLflow, dan LIME.")
print("- Performa model dievaluasi menggunakan MSE, RMSE, MAE, dan R2.")
print("- Notebook ini dibuat aman untuk Google Colab dengan menghindari notebook widget interaktif yang rawan merusak file .ipynb.")